> 📅 __Date: 2026-08-31__

# 🛠️ **RAG Implementation**

> **Goal:** Implement a complete Retrieval-Augmented Generation (RAG) pipeline in LangChain — from PDF loading and chunking to embeddings, vector storage, retrieval, prompt augmentation, generation, and finally a higher-level `RetrievalQA` chain.

> **Prerequisite:** The previous RAG Architecture notes explain the concepts. This chapter focuses on the **actual implementation**.

---

# 🗺️ **RAG Implementation Roadmap**

```text
PDF
 ↓
Text Extraction
 ↓
Chunking
 ↓
Embedding Model
 ↓
Vector Database
 ↓
Indexing / Ingestion
 ↓
Retriever
 ↓
Relevant Chunks
 ↓
Context Creation
 ↓
Prompt / Augmentation
 ↓
LLM / Generation
 ↓
Answer
```

**The implementation can be divided into two main pipelines:**

```text
1. Ingestion / Indexing Pipeline
2. Runtime / Query Pipeline
```

---

# 🎯 **Project**

## **Build a Q&A System on the "Attention Is All You Need" Paper**

**Knowledge source:**

```text
assets/NIPS-2017-attention-is-all-you-need-Paper.pdf
```

**Example user query:**

```text
"What is positional encoding?"
```

**The complete RAG system should:**

```text
User Query
   ↓
Find Relevant Information
   ↓
Give Relevant Context to LLM
   ↓
Generate Answer
```

---

# 📦 **1. Install Required Packages**

**For PDF loading and document processing:**

```python
%pip install -U langchain-community pypdf
```

**For the OpenAI integration:**

```python
%pip install -U langchain-openai
```

**For Chroma:**

```python
%pip install -U chromadb
```

**The important packages are:**

```text
langchain-community
pypdf
langchain-openai
chromadb
```

---

# 📥 **Stage 1 — Text Extraction**

> **Text Extraction = Loading the source document and extracting its content into LangChain `Document` objects.**

For a PDF, we use **`PyPDFLoader`**.

In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    "assets/NIPS-2017-attention-is-all-you-need-Paper.pdf"
)

docs = loader.load()

C:\Users\KP\AppData\Local\Temp\ipykernel_11296\3244149555.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


**The flow is:**

```text
PDF
 ↓
PyPDFLoader
 ↓
docs
 ↓
List of Document Objects
```

---

# 🔍 **Inspect the Loaded Documents**

In [2]:
type(docs)

list

**Typically:**

```text
list
```

**Inspect the first item:**

In [3]:
type(docs[0])

langchain_core.documents.base.Document

**Typically:**

```text
Document
```

**Inspect its content:**

```python
docs[0].page_content
```

**Inspect metadata:**

```python
docs[0].metadata
```

**Conceptually:**

```text
Document
 ├── page_content
 └── metadata
```

---

# ✂️ **Stage 2 — Chunking**

> **Chunking = Splitting a large document into smaller pieces that can be embedded, stored and retrieved independently.**

**We use `RecursiveCharacterTextSplitter`:**

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

**Then:**

In [5]:
chunks = text_splitter.split_documents(docs)

**Conceptually:**

```text
Documents
   ↓
RecursiveCharacterTextSplitter
   ↓
Chunks
```

---

# 📏 **Chunk Size**

In [6]:
chunk_size = 1000

> **`chunk_size` = Maximum target size of each chunk in this character-based splitter configuration.**

**Example:**

```text
Chunk 1 → ~1000 characters
Chunk 2 → ~1000 characters
Chunk 3 → ~1000 characters
...
```

> **Important:** `chunk_size=1000` here refers to the splitter's text-size setting, not 1000 tokens.

---

# 🔄 **Chunk Overlap**

In [7]:
chunk_overlap = 200

> **`chunk_overlap` = The amount of text shared between consecutive chunks.**

Why do we need overlap?

```text
Chunk 1
────────────────────────────
1 ................. 1000
          ↓
       overlap
          ↓
Chunk 2
────────────────────────────
801 .............. 1800
```

**A simplified example is:**

```text
1 - 1000
801 - 1800
1601 - 2600
2401 - 3400
...
```

**The overlap can be understood as:**

$$
\text{Next Start}
\approx
\text{Current Start} + \text{chunk\_size} - \text{chunk\_overlap}
$$

**For this configuration:**

$$
1000 - 200 = 800
$$

so the next chunk begins approximately 800 characters after the current chunk's start.

> **The exact boundaries can depend on the splitter's separator hierarchy and the actual text.**

---

# 🧠 **Why Chunk Overlap Matters**

**A concept may cross a chunk boundary:**

```text
Chunk 1:
"The Transformer uses positional..."

Chunk 2:
"...encodings to inject information..."
```

Overlap keeps some shared context between the chunks.

> **Chunk overlap helps reduce information loss at chunk boundaries.**

---

# 🆚 **Chunk Size vs Chunk Overlap**

| Parameter | Meaning |
|---|---|
| **chunk_size** | Maximum target size of a chunk |
| **chunk_overlap** | Shared content between consecutive chunks |

**Memory trick:**

```text
chunk_size
→ How big?

chunk_overlap
→ How much is shared?
```

---

# 📊 **Inspect the Chunks**

In [8]:
len(chunks)

43

**Inspect a chunk:**

In [9]:
chunks[0].page_content

'Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence and convolutions\nentirely. Experiments on two machine translation tasks show these models to\nbe superior in quality while being more parallelizable and requiring signiﬁcantly

**Inspect metadata:**

In [10]:
chunks[0].metadata

{'producer': 'PyPDF2',
 'creator': 'PyPDF',
 'creationdate': '',
 'subject': 'Neural Information Processing Systems http://nips.cc/',
 'publisher': 'Curran Associates, Inc.',
 'language': 'en-US',
 'created': '2017',
 'eventtype': 'Poster',
 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-to-

---

# 🔐 **Stage 3 — Configure OpenAI API Key**

The embedding model and LLM in this implementation use OpenAI.

## ☁️ **Google Colab**

```python
from google.colab import userdata

openai = userdata.get("OPENAI")

import os

os.environ["OPENAI_API_KEY"] = openai
```

**The flow is:**

```text
Colab Secret
    ↓
OPENAI_API_KEY
    ↓
LangChain OpenAI Integration
```

## 💻 **VS Code / Local IDE**

Use a `.env` file:

```dotenv
OPENAI_API_KEY=your_api_key_here
```

**Then:**

In [11]:
from dotenv import load_dotenv

load_dotenv()

True

> **Never print or commit the actual API key.**

---

# 🧠 **Stage 4 — Embedding Model**

> **Embedding Model = Converts text into numerical vectors that can be compared for semantic similarity.**

In [12]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings()

**Conceptually:**

```text
Chunk
 ↓
Embedding Model
 ↓
Vector
```

**For every chunk:**

```text
Chunk 1 → Vector 1
Chunk 2 → Vector 2
Chunk 3 → Vector 3
...
```

The same embedding model is used conceptually for the user's query during retrieval.

```text
User Query
 ↓
Embedding Model
 ↓
Query Vector
```

---

# 🗃️ **Stage 5 — Vector Database**

We use **Chroma** as the vector store.

In [13]:
from langchain_community.vectorstores import Chroma

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="db"
)

**Conceptually:**

```text
Chunks
  ↓
Embedding Model
  ↓
Vectors
  ↓
Chroma
```

---

# 🧩 **What Does `Chroma.from_documents()` Do?**

**Conceptually:**

```text
Document Chunks
      ↓
Generate Embeddings
      ↓
Store Vectors
      +
Store Document Content
      +
Store Metadata
      ↓
Chroma Vector Store
```

So after this step, we have a searchable knowledge base.

---

# 💾 **Persistence**

**The implementation uses:**

In [14]:
persist_directory="db"

This specifies a local directory for the Chroma store.

**Conceptually:**

```text
Application
    ↓
Chroma
    ↓
db/
```

> **Persistence allows vector-store data to be kept outside the immediate Python process.**

---

# 🏗️ **Complete Indexing / Ingestion Pipeline**

**The complete ingestion flow is:**

```text
Document
   ↓
Loader
   ↓
Extracted Text
   ↓
Chunking
   ↓
Chunk 1, Chunk 2, Chunk 3, ...
   ↓
Embedding Model
   ↓
Vectors
   ↓
Vector Database
```

**This is called:**

> **Indexing / Ingestion Pipeline**

---

# 🧭 **Indexing Pipeline**

```text
                    DOCUMENT
                       │
                       ↓
                DOCUMENT LOADER
                       │
                       ↓
                DOCUMENT OBJECTS
                       │
                       ↓
                    CHUNKING
                       │
              ┌────────┼────────┐
              ↓        ↓        ↓
           Chunk 1  Chunk 2  Chunk 3  ...
              │        │        │
              └────────┼────────┘
                       ↓
                 EMBEDDING MODEL
                       ↓
                     VECTORS
                       ↓
                  CHROMA / DB
```

---

# 🧩 **Reusable Ingestion Function**

**Instead of repeating the indexing steps, wrap them in a function:**

In [15]:
def ingestion_pipeline(doc_path):
    loader = PyPDFLoader(doc_path)
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
    )

    chunks = text_splitter.split_documents(docs)

    vector_db = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory="db"
    )

    return vector_db

**Then:**

In [16]:
vector_db = ingestion_pipeline(
    "assets/NIPS-2017-attention-is-all-you-need-Paper.pdf"
)

---

# 🔎 **Stage 6 — Retriever**

**Now create the retriever:**

In [17]:
retriever = vector_db.as_retriever()

**Conceptually:**

```text
Vector DB
    ↓
Retriever
```

---

# ❓ **User Query**

In [18]:
user_query = "What is positional encoding?"

**Retrieve relevant chunks:**

In [19]:
relevant_chunks = retriever.invoke(user_query)

**The flow is:**

```text
User Query
      ↓
Retriever
      ↓
Relevant Chunks
```

---

# 🔍 **Inspecting a Retrieved Chunk**

In [20]:
relevant_chunks[0]

Document(metadata={'book': 'Advances in Neural Information Processing Systems 30', 'eventtype': 'Poster', 'moddate': '2018-02-12T21:22:10-08:00', 'published': '2017', 'date': '2017', 'lastpage': '6008', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-to-French translation, we outperform the p

**Text:**

In [21]:
relevant_chunks[0].page_content

'Similarly to other sequence transduction models, we use learned embeddings to convert the input\ntokens and output tokens to vectors of dimensiondmodel. We also use the usual learned linear transfor-\nmation and softmax function to convert the decoder output to predicted next-token probabilities. In\nour model, we share the same weight matrix between the two embedding layers and the pre-softmax\nlinear transformation, similar to [24]. In the embedding layers, we multiply those weights by√dmodel.\n3.5 Positional Encoding\nSince our model contains no recurrence and no convolution, in order for the model to make use of the\norder of the sequence, we must inject some information about the relative or absolute position of the\ntokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the\n5'

**Metadata:**

In [22]:
relevant_chunks[0].metadata

{'book': 'Advances in Neural Information Processing Systems 30',
 'eventtype': 'Poster',
 'moddate': '2018-02-12T21:22:10-08:00',
 'published': '2017',
 'date': '2017',
 'lastpage': '6008',
 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-to-French translation, we outperform the previoussingl

---

# 📊 **How Many Chunks are Retrieved?**

In [23]:
len(
    retriever.invoke(
        "What is positional encoding?"
    )
)

4

In the example configuration, the default retriever returns **4 documents**.

> **Important:** `4` is a default/example configuration here, not a universal RAG rule. The number of retrieved documents can be configured.

---

# 🧠 **What Happens During Retrieval?**

**Conceptually:**

```text
User Query
      ↓
Query Embedding
      ↓
Query Vector
      ↓
Similarity Search
      ↓
Stored Chunk Vectors
      ↓
Relevant Chunks
```

The vector store and retriever integration can handle the embedding and search operations internally.

---

# 🧱 **Create the Context**

**Combine the retrieved chunk text:**

In [24]:
context = ""

for chunk in relevant_chunks:
    context += chunk.page_content

**Flow:**

```text
Relevant Chunks
      ↓
Extract page_content
      ↓
Combine Text
      ↓
Context
```

**A cleaner equivalent is:**

In [25]:
context = "\n\n".join(
    chunk.page_content
    for chunk in relevant_chunks
)

---

# ✨ **Stage 7 — Prompt Creation / Augmentation**

**Now combine the retrieved context with the user's query:**

In [26]:
prompt = f"""
Based on the given context, answer the question.

Context:
{context}

Question:
{user_query}
"""

This is the **augmentation** stage.

---

# 🔄 **Augmentation Flow**

```text
Relevant Chunks
      ↓
Context
      +
User Query
      ↓
Prompt
```

**Conceptually:**

$$
\text{Augmented Prompt}
=
f(\text{Retrieved Context},\text{User Query})
$$

---

# 🤖 **Stage 8 — Generation**

**Create the model and send the prompt:**

In [27]:
from langchain_openai import OpenAI

model = OpenAI()

response = model.invoke(prompt)

print(response)


Positional encoding is a technique used in sequence transduction models to inject information about the relative or absolute positions of tokens in a sequence. It involves adding a "positional encoding" to input embeddings, where each dimension corresponds to a sinusoid with specific wavelengths. This allows the model to learn to attend to relative positions within a sequence.


**Flow:**

```text
Augmented Prompt
       ↓
      LLM
       ↓
    Response
```

---

# 🏗️ **Runtime Pipeline**

**The runtime pipeline is:**

```text
Query
   ↓
Retriever
   ↓
Relevant Chunks
   ↓
Context
   ↓
Prompt
   ↓
LLM
   ↓
Response
```

**At a lower level:**

```text
Query
  ↓
Embedding Model
  ↓
Query Vector
  ↓
Similarity Search
  ↓
Relevant Chunks
  ↓
Query + Context
  ↓
LLM
  ↓
Response
```

---

# 🧩 **Reusable Runtime Function**

**Instead of repeating the runtime steps:**

In [28]:
def runtime_pipeline(query):
    relevant_chunks = retriever.invoke(query)

    context = ""

    for chunk in relevant_chunks:
        context += chunk.page_content

    prompt = f"""
    Based on the given context, answer the question.

    Context:
    {context}

    Question:
    {query}
    """

    return model.invoke(prompt)

**Then:**

In [29]:
response = runtime_pipeline(
    "What is positional encoding?"
)

print(response)


Positional encoding is a method used in sequence transduction models to inject information about the relative or absolute position of tokens in a sequence. It involves adding "positional encodings" to the input embeddings, where each dimension corresponds to a sinusoid with different wavelengths. This allows the model to easily learn to attend by relative positions, making it useful for tasks involving long sequences. 


---

# 🧠 **Why Create a Runtime Function?**

**Instead of repeating:**

```text
Retrieve
↓
Create Context
↓
Create Prompt
↓
Call Model
```

**we encapsulate the workflow in:**

```python
runtime_pipeline(query)
```

**Advantages:**

```text
Reusable
Cleaner
Easier to Test
Easier to Modify
```

---

# 🏗️ **Complete RAG Implementation**

```text
                    INDEXING / INGESTION
                           │
Document ─→ Loader ─→ Chunking ─→ Embeddings ─→ Chroma
                                                   │
                                                   ↓
                    RUNTIME                  Similarity Search
                                                   ↑
                                                   │
Query ─→ Query Embedding ─→ Query Vector ─────────┘
                                                   │
                                                   ↓
                                            Relevant Chunks
                                                   │
                                                   ↓
                                      Context + User Query
                                                   │
                                                   ↓
                                               Prompt
                                                   │
                                                   ↓
                                                  LLM
                                                   │
                                                   ↓
                                                Answer
```

---

# 🗂️ **Indexing Pipeline vs Runtime Pipeline**

| Pipeline | Steps | When It Runs |
|---|---|---|
| **Indexing / Ingestion** | Load → Chunk → Embed → Store | Before / while preparing the knowledge base |
| **Runtime / Query** | Query → Retrieve → Augment → Generate | Every time the user asks a question |

**Memory trick:**

```text
INDEXING
→ Prepare the knowledge

RUNTIME
→ Use the knowledge
```

---

# 🧠 **Indexing Usually Happens Less Frequently**

The indexing pipeline is generally not repeated for every question.

**Instead:**

```text
Documents
   ↓
Index Once
   ↓
Vector DB
```

**Then multiple queries can use the same index:**

```text
Query 1 ─┐
Query 2 ─┤
Query 3 ─┼→ Retriever → LLM
Query 4 ─┤
Query 5 ─┘
```

This separation is one of the most important ideas in RAG architecture.

---

# 🔄 **Complete Example — Manual RAG Pipeline**

In [31]:
# -----------------------------
# 1. Text Extraction
# -----------------------------

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    "assets/NIPS-2017-attention-is-all-you-need-Paper.pdf"
)

docs = loader.load()


# -----------------------------
# 2. Chunking
# -----------------------------

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = text_splitter.split_documents(docs)


# -----------------------------
# 3. Embedding Model
# -----------------------------

from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings()


# -----------------------------
# 4. Vector Database
# -----------------------------

from langchain_community.vectorstores import Chroma

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="db"
)


# -----------------------------
# 5. Retriever
# -----------------------------

retriever = vector_db.as_retriever()


# -----------------------------
# 6. User Query
# -----------------------------

user_query = "What is positional encoding?"

relevant_chunks = retriever.invoke(
    user_query
)


# -----------------------------
# 7. Context Creation
# -----------------------------

context = ""

for chunk in relevant_chunks:
    context += chunk.page_content


# -----------------------------
# 8. Prompt / Augmentation
# -----------------------------

prompt = f"""
Based on the given context, answer the question.

Context:
{context}

Question:
{user_query}
"""


# -----------------------------
# 9. Generation
# -----------------------------

from langchain_openai import OpenAI

model = OpenAI()

response = model.invoke(prompt)

print(response)


Positional encoding is a method used in models that contain no recurrence and no convolution to inject information about the relative or absolute position of tokens in a sequence. This is accomplished by adding "positional encodings" to the input embeddings, which correspond to sinusoids with wavelengths ranging from 2π to 10000· 2π. This allows the model to learn to attend by relative positions, and may also help with extrapolating to longer sequence lengths.


---

# 🔄 **Manual RAG Flow**

```text
PDF
 ↓
PyPDFLoader
 ↓
Documents
 ↓
RecursiveCharacterTextSplitter
 ↓
Chunks
 ↓
OpenAIEmbeddings
 ↓
Vectors
 ↓
Chroma
 ↓
Retriever
 ↓
Relevant Chunks
 ↓
Context
 ↓
Prompt
 ↓
OpenAI Model
 ↓
Response
```

---

# 🔗 **High-Level `RetrievalQA`**

The manual pipeline makes every RAG step explicit.

LangChain also provides higher-level chain abstractions for some retrieval workflows.

**The example uses the classic `RetrievalQA` API:**

In [32]:
from langchain_classic.chains import RetrievalQA

**Create the chain:**

In [33]:
chain = RetrievalQA.from_chain_type(
    llm=model,
    retriever=retriever,
    return_source_documents=True
)

**Then:**

In [34]:
response = chain.invoke(
    "What is positional encoding?"
)

print(response)

{'query': 'What is positional encoding?', 'result': ' Positional encoding is a technique used in sequence transduction models to inject information about the relative or absolute position of tokens in a sequence. It involves adding sinusoidal functions to input embeddings, allowing the model to easily learn to attend by relative positions.', 'source_documents': [Document(metadata={'moddate': '2018-02-12T21:22:10-08:00', 'page': 4, 'language': 'en-US', 'created': '2017', 'firstpage': '5998', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'book': 'Advances in Neural Information Processing Systems 30', 'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett', 'creationdate': '', 'type': 'Conference Proceedings', 'title': 'Attention is All you Need', 'description-abstract': 'The dominant sequence transduction models are based on complex recurren

---

# 🧩 **What Does `RetrievalQA` Do?**

**Conceptually, it packages a retrieval-based question-answering workflow:**

```text
Question
   ↓
Retriever
   ↓
Relevant Documents
   ↓
Context / Prompt
   ↓
LLM
   ↓
Answer
```

**Instead of manually writing:**

```text
Retrieve
+
Create Context
+
Create Prompt
+
Call Model
```

the chain provides a higher-level abstraction.

---

# 📚 **`return_source_documents=True`**

In [35]:
return_source_documents=True

This asks the chain to include the retrieved source documents in its result.

**Conceptually:**

```text
Question
   ↓
Retriever
   ↓
Source Documents
   ↓
LLM
   ↓
Answer + Sources
```

This is useful when we want to inspect or display where the answer came from.

---

# 🔍 **Inspecting the `RetrievalQA` Result**

In [36]:
response = chain.invoke(
    "What is positional encoding?"
)

response

{'query': 'What is positional encoding?',
 'result': ' Positional encoding is a technique used in sequence transduction models to inject information about the relative or absolute position of tokens in a sequence. This is necessary for models that do not use recurrence or convolution, as it allows them to make use of the order of the sequence in their predictions. It involves adding "positional encodings" to the input embeddings, which are sinusoidal functions with wavelengths that form a geometric progression from 2π to 10000· 2π. These positional encodings allow the model to easily learn to attend by relative positions, and can be extrapolated to longer sequences during training. ',
 'source_documents': [Document(metadata={'lastpage': '6008', 'language': 'en-US', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the enco

**We can inspect the available keys:**

In [37]:
response.keys()

dict_keys(['query', 'result', 'source_documents'])

**For classic `RetrievalQA`, the result commonly contains:**

```text
result
source_documents
```

**So we can inspect:**

In [38]:
response["result"]

' Positional encoding is a technique used in sequence transduction models to inject information about the relative or absolute position of tokens in a sequence. This is necessary for models that do not use recurrence or convolution, as it allows them to make use of the order of the sequence in their predictions. It involves adding "positional encodings" to the input embeddings, which are sinusoidal functions with wavelengths that form a geometric progression from 2π to 10000· 2π. These positional encodings allow the model to easily learn to attend by relative positions, and can be extrapolated to longer sequences during training. '

**and:**

In [39]:
response["source_documents"]

[Document(metadata={'lastpage': '6008', 'language': 'en-US', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-to-French translation, we outperform the previoussingle state-of-the-art with model by 0.7 BLEU, achieving a BLEU score of 41.1.', 'book': 'Advances in Neural Information Processing Sy

> **The exact result schema can vary across versions, so inspecting `response.keys()` is a good practice.**

---

# 🆚 **Manual RAG vs `RetrievalQA`**

| Area | Manual RAG | `RetrievalQA` |
|---|---|---|
| Retrieval | Explicit | Encapsulated |
| Context creation | Explicit | Encapsulated |
| Prompt construction | Explicit | Encapsulated |
| LLM call | Explicit | Encapsulated |
| Learning value | High | High-level abstraction |
| Customization | Greater | More constrained by chain API |
| Debugging | Easy to inspect each stage | More abstraction |
| Architecture visibility | Very clear | More hidden |

**Memory trick:**

```text
Manual RAG
→ Build every step yourself

RetrievalQA
→ Package common retrieval + QA steps
```

---

# ⚠️ **Legacy API Note**

**The example uses:**

```python
from langchain_classic.chains import RetrievalQA
```

This is a **classic / legacy-style chain API**.

For learning, it is useful because it clearly shows the abstraction:

```text
Retriever
+
LLM
=
Retrieval-based Q&A Chain
```

**However:**

> **For new applications, check the current LangChain documentation before choosing `RetrievalQA`, because LangChain's APIs and recommended composition patterns evolve over time.**

The manual pipeline remains valuable because it exposes the underlying architecture.

---

# 🧠 **Manual RAG is Better for Learning**

**When learning RAG, understand each stage individually:**

```text
1. Load
2. Split
3. Embed
4. Store
5. Retrieve
6. Build Context
7. Build Prompt
8. Generate
```

Only then move to higher-level abstractions.

---

# 🧪 **Inspecting Retrieval Results**

**For debugging, inspect retrieved chunks before sending them to the LLM:**

In [40]:
relevant_chunks = retriever.invoke(
    "What is positional encoding?"
)

for i, chunk in enumerate(relevant_chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content)
    print("Metadata:", chunk.metadata)


--- Chunk 1 ---
Similarly to other sequence transduction models, we use learned embeddings to convert the input
tokens and output tokens to vectors of dimensiondmodel. We also use the usual learned linear transfor-
mation and softmax function to convert the decoder output to predicted next-token probabilities. In
our model, we share the same weight matrix between the two embedding layers and the pre-softmax
linear transformation, similar to [24]. In the embedding layers, we multiply those weights by√dmodel.
3.5 Positional Encoding
Since our model contains no recurrence and no convolution, in order for the model to make use of the
order of the sequence, we must inject some information about the relative or absolute position of the
tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the
5
Metadata: {'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'total_pages': 11, 'publish

**This helps answer:**

```text
Did retrieval return the correct information?
```

---

# 🔬 **Debugging a RAG Pipeline**

**A useful debugging strategy is:**

```text
Query
 ↓
Check Query
 ↓
Retriever
 ↓
Check Retrieved Chunks
 ↓
Context
 ↓
Check Prompt
 ↓
LLM
 ↓
Check Answer
```

**If the final answer is wrong, the failure may be in:**

```text
Chunking
Retrieval
Embedding
Context
Prompt
Generation
```

> **Do not immediately assume the LLM is the source of the problem.**

---

# 🎯 **Retrieval Quality Before Generation Quality**

**A useful principle is:**

```text
Bad Retrieval
      ↓
Bad Context
      ↓
LLM
      ↓
Bad Answer
```

**Whereas:**

```text
Good Retrieval
      ↓
Relevant Context
      ↓
LLM
      ↓
Better Chance of Correct Answer
```

**Therefore:**

> **Always inspect retrieval quality when debugging a RAG application.**

---

# 🧠 **Why Chunking + Retrieval Work Together**

**Chunking alone:**

```text
Document
 ↓
Chunks
```

does not answer the user's question.

Retrieval needs searchable chunks to operate effectively.

**Together:**

```text
Document
 ↓
Chunking
 ↓
Embeddings
 ↓
Vector DB
 ↓
Retriever
 ↓
Relevant Chunks
```

**Therefore:**

> **Chunking creates the searchable units; retrieval selects the useful units.**

---

# 🔗 **Embedding and Retrieval Relationship**

### **During Indexing**

```text
Document Chunk
      ↓
Embedding Model
      ↓
Chunk Vector
      ↓
Vector DB
```

### **During Runtime**

```text
User Query
      ↓
Embedding Model
      ↓
Query Vector
      ↓
Vector DB Search
```

This allows the query and stored chunks to be compared in the same vector space.

---

# 🧮 **Similarity Search**

**Let:**

```text
q = Query vector
d_i = Chunk vector i
```

**Then cosine similarity can be written as:**

$$
\operatorname{sim}(q,d_i)
=
\frac{q\cdot d_i}
{\|q\|\|d_i\|}
$$

The retriever can rank chunks according to their similarity scores.

**Conceptually:**

```text
Chunk 1 → 0.91
Chunk 2 → 0.84
Chunk 3 → 0.71
Chunk 4 → 0.52
```

**Then:**

```text
Top-k
 ↓
Relevant Chunks
```

> **The actual scoring method depends on the vector store and retrieval configuration; cosine similarity is the conceptual model used here.**

---

# 🧠 **RAG Does Not Mean Only Vector Search**

This implementation uses embedding-based retrieval, **but retrieval systems can use other strategies too:**

```text
Vector Similarity Search
Keyword Search
Hybrid Search
Metadata Filtering
Reranking
```

**For this notebook, the core pattern is:**

```text
Embedding
+
Chroma
+
Similarity Retrieval
```

---

# 🏗️ **Production-Oriented RAG View**

The notebook implementation is intentionally simple.

**A more complete production system may add:**

```text
Document Cleaning
+
Better Chunking
+
Metadata
+
Embedding Model
+
Vector DB
+
Retriever
+
Reranker
+
Prompt Template
+
LLM
+
Citations
+
Evaluation
+
Observability
```

**Conceptually:**

```text
Knowledge
 ↓
Indexing
 ↓
Vector DB
 ↓
Retriever
 ↓
Reranker
 ↓
Prompt
 ↓
LLM
 ↓
Answer + Sources
```

---

# 🆚 **Architecture vs Implementation**

The RAG architecture tells us **what components exist**.

The implementation tells us **how those components are connected in code**.

### **Architecture**

```text
Document
 ↓
Chunking
 ↓
Embedding
 ↓
Vector DB
 ↓
Retriever
 ↓
Augmentation
 ↓
LLM
```

### **Implementation**

```python
loader = PyPDFLoader(...)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(...)
chunks = text_splitter.split_documents(docs)

embedding_model = OpenAIEmbeddings()

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="db"
)

retriever = vector_db.as_retriever()
relevant_chunks = retriever.invoke(query)

model.invoke(prompt)
```

---

# 🔄 **Complete Flow in One Diagram**

```text
                         OFFLINE / INDEXING
                                │
                                ↓
                         PDF / DOCUMENT
                                │
                                ↓
                              LOADER
                                │
                                ↓
                           DOCUMENTS
                                │
                                ↓
                             CHUNKING
                                │
                                ↓
                       EMBEDDING MODEL
                                │
                                ↓
                             VECTORS
                                │
                                ↓
                           CHROMA DB
                                │
                                │
                                ↓
                         ONLINE / RUNTIME
                                │
                           USER QUERY
                                ↓
                        QUERY EMBEDDING
                                ↓
                        SIMILARITY SEARCH
                                ↓
                        RELEVANT CHUNKS
                                ↓
                           CONTEXT
                                ↓
                    QUERY + RETRIEVED CONTEXT
                                ↓
                             PROMPT
                                ↓
                               LLM
                                ↓
                             ANSWER
```

---

# 🏁 **Complete RAG Mental Model**

```text
                         RAG IMPLEMENTATION
                                 │
             ┌───────────────────┴───────────────────┐
             ↓                                       ↓
          INDEXING                                  RUNTIME
             │                                       │
             ↓                                       ↓
        PDF / DATA                               USER QUERY
             ↓                                       ↓
          LOADER                               EMBEDDING MODEL
             ↓                                       ↓
       DOCUMENT OBJECTS                         QUERY VECTOR
             ↓                                       ↓
         CHUNKING                              RETRIEVER / SEARCH
             ↓                                       ↓
        EMBEDDING MODEL                         RELEVANT CHUNKS
             ↓                                       ↓
           VECTORS                                  CONTEXT
             ↓                                       ↓
          CHROMA                            QUERY + CONTEXT
                                                     ↓
                                                  PROMPT
                                                     ↓
                                                    LLM
                                                     ↓
                                                  ANSWER
```

---

# 📋 **Quick Revision**

```text
PyPDFLoader
→ Loads PDF into Document objects

Document
→ Contains page_content and metadata

RecursiveCharacterTextSplitter
→ Splits documents into chunks

chunk_size
→ Maximum target chunk size in this configuration

chunk_overlap
→ Shared content between consecutive chunks

OpenAIEmbeddings
→ Converts text into vectors

Chroma
→ Vector store used to store and search embeddings

persist_directory
→ Local persistence location for Chroma data

Ingestion / Indexing
→ Load → Chunk → Embed → Store

Retriever
→ Retrieves relevant documents

as_retriever()
→ Creates a retriever interface from the vector store

invoke(query)
→ Executes the retriever for a query

Context
→ Combined text from retrieved documents

Augmentation
→ Combines context with user query

LLM
→ Generates final answer

Runtime Pipeline
→ Query → Retrieve → Context → Prompt → LLM

RetrievalQA
→ Higher-level classic chain for retrieval-based question answering
```

---

# 🧠 **Memory Trick**

```text
LOAD
→ Get the document

SPLIT
→ Break it into chunks

EMBED
→ Convert chunks into vectors

STORE
→ Put vectors into Vector DB

RETRIEVE
→ Find relevant chunks

AUGMENT
→ Add chunks to query

GENERATE
→ Ask the LLM
```

**The shortest implementation formula is:**

```text
RAG
=
Load
+
Chunk
+
Embed
+
Store
+
Retrieve
+
Augment
+
Generate
```

---

# 🧭 **Indexing vs Runtime — Memory Trick**

```text
INDEXING
────────────────────
Document
 ↓
Loader
 ↓
Chunking
 ↓
Embedding
 ↓
Vector DB


RUNTIME
────────────────────
Query
 ↓
Embedding
 ↓
Retriever
 ↓
Relevant Chunks
 ↓
Prompt
 ↓
LLM
 ↓
Answer
```

---

# 🎓 **Interview-Friendly Explanation**

> **If an interviewer asks: "How would you implement RAG in LangChain?"**

```text
First, I load the source document using a document loader such as
PyPDFLoader.

Then I split the documents into smaller chunks using a text splitter,
for example RecursiveCharacterTextSplitter with a configured
chunk size and overlap.

Next, I convert those chunks into embeddings using an embedding model
and store them in a vector database such as Chroma.

At runtime, when the user asks a question, I pass the query to a
retriever. The retriever searches the vector database and returns
the most relevant chunks.

I then combine those retrieved chunks with the user's question to
create an augmented prompt.

Finally, I send that prompt to the LLM, which generates the answer.

So the overall flow is:
Load → Chunk → Embed → Store → Retrieve → Augment → Generate.
```

---

# 🧠 **Interview: Why Do We Need Chunking?**

```text
We use chunking because sending an entire large document to the LLM
can exceed its context window.

Chunking also lets the application retrieve only the relevant pieces
instead of passing the complete document every time.
```

---

# 🧠 **Interview: Why Do We Need Embeddings?**

```text
The user query and document chunks are initially text,
but similarity search works on numerical representations.

So we convert both the query and document chunks into vectors
using an embedding model.

Then we can compare the vectors using a similarity measure
such as cosine similarity and retrieve relevant chunks.
```

---

# 🧠 **Interview: Why Do We Need a Vector Database?**

```text
A vector database provides a place to store embeddings,
along with the original text and metadata, and search for
vectors that are similar to the query vector.
```

---

# 🧠 **Interview: What Happens During Runtime?**

```text
The user sends a query.

The query is converted into an embedding and compared with the
stored document embeddings.

The retriever returns the most relevant chunks.

Those chunks are combined with the query to create the prompt,
and the LLM generates the final answer.
```

---

# ⚠️ **Common Beginner Mistakes**

## **1. Embedding Only the Documents**

RAG also needs a query embedding during retrieval.

```text
Document Chunks
 ↓
Embeddings
```

**and:**

```text
User Query
 ↓
Query Embedding
```

---

## **2. Treating Chunk Size as Token Count**

**In this implementation:**

```python
chunk_size=1000
```

is the splitter's text-size setting.

**Do not automatically interpret it as:**

```text
1000 tokens
```

---

## **3. Forgetting Chunk Overlap**

Overlap helps preserve shared context across boundaries.

```text
Chunk 1
   ↓
Shared Context
   ↓
Chunk 2
```

---

## **4. Passing Every Chunk to the LLM**

The purpose of retrieval is to select relevant information.

```text
All Chunks
   ↓
❌ Huge Context
```

**Instead:**

```text
All Chunks
   ↓
Retriever
   ↓
Relevant Chunks
   ↓
LLM
```

---

## **5. Assuming the Retriever Always Returns 4 Chunks**

**In this example:**

```python
retriever = vector_db.as_retriever()
```

the default configuration returns four documents.

But retrieval configuration can be changed.

> **`k=4` is an example configuration, not a universal RAG rule.**

---

## **6. Not Inspecting Retrieved Chunks**

**When the answer is wrong, first inspect:**

```python
relevant_chunks
```

The problem may be retrieval rather than generation.

---

## **7. Treating `RetrievalQA` as Magic**

**A high-level chain hides several steps:**

```text
Retrieve
+
Context
+
Prompt
+
Generate
```

Understanding the manual implementation makes the abstraction much easier to understand.

---

# 🔬 **Debugging Checklist**

```text
1. Is the document loaded correctly?
        ↓
2. Are the chunks meaningful?
        ↓
3. Are embeddings being created?
        ↓
4. Is the vector DB populated?
        ↓
5. Is the retriever returning relevant chunks?
        ↓
6. Is the context correct?
        ↓
7. Is the prompt clear?
        ↓
8. Is the LLM generating correctly?
```

> **Do not debug the entire RAG pipeline as one black box. Check every stage separately.**

---

# 🏗️ **RAG Implementation Responsibilities**

| Stage | Code / Component | Responsibility |
|---|---|---|
| **Extraction** | `PyPDFLoader` | Load PDF content |
| **Chunking** | `RecursiveCharacterTextSplitter` | Split documents |
| **Embedding** | `OpenAIEmbeddings` | Convert text to vectors |
| **Storage** | `Chroma` | Store / search vectors and documents |
| **Retrieval** | `vector_db.as_retriever()` | Fetch relevant chunks |
| **Augmentation** | Prompt construction | Combine context + query |
| **Generation** | `OpenAI` | Generate the answer |
| **High-level QA** | `RetrievalQA` | Encapsulate retrieval + generation |

---

# 🆚 **Manual Pipeline vs High-Level Chain**

```text
MANUAL RAG
─────────────────────────────────────

Loader
  ↓
Chunker
  ↓
Embedding
  ↓
Vector DB
  ↓
Retriever
  ↓
Context
  ↓
Prompt
  ↓
LLM
  ↓
Answer
```

```text
HIGH-LEVEL RETRIEVAL QA
─────────────────────────────────────

Retriever
    +
LLM
    ↓
RetrievalQA
    ↓
Answer + Sources
```

The manual implementation is useful for understanding exactly what happens.

The high-level chain is useful when an abstraction already matches the required workflow.

---

# 🧠 **Complete RAG Mental Model**

```text
                         RAG IMPLEMENTATION
                                 │
             ┌───────────────────┴───────────────────┐
             ↓                                       ↓
          INDEXING                                  RUNTIME
             │                                       │
             ↓                                       ↓
        PDF / DATA                               USER QUERY
             ↓                                       ↓
          LOADER                               EMBEDDING MODEL
             ↓                                       ↓
       DOCUMENT OBJECTS                         QUERY VECTOR
             ↓                                       ↓
         CHUNKING                              RETRIEVER / SEARCH
             ↓                                       ↓
        EMBEDDING MODEL                         RELEVANT CHUNKS
             ↓                                       ↓
           VECTORS                                  CONTEXT
             ↓                                       ↓
          CHROMA                            QUERY + CONTEXT
                                                     ↓
                                                  PROMPT
                                                     ↓
                                                    LLM
                                                     ↓
                                                  ANSWER
```

---

# 🏁 **Key Takeaways**

> **1. RAG implementation has two major phases: indexing and runtime.**

> **2. Indexing prepares the knowledge base; runtime uses that knowledge to answer queries.**

> **3. `PyPDFLoader` loads the PDF into LangChain `Document` objects.**

> **4. `RecursiveCharacterTextSplitter` divides documents into smaller chunks.**

> **5. `chunk_size` controls the target chunk size, while `chunk_overlap` preserves shared context between consecutive chunks.**

> **6. Embedding models convert both document chunks and user queries into vectors.**

> **7. Chroma can store embeddings together with document content and metadata for retrieval.**

> **8. A retriever searches the vector store and returns relevant chunks.**

> **9. Retrieved chunks are combined with the user query during augmentation.**

> **10. The LLM uses the augmented prompt to generate the final answer.**

> **11. The manual RAG implementation makes every stage visible and is valuable for learning and debugging.**

> **12. `RetrievalQA` is a higher-level classic abstraction for retrieval-based question answering.**

> **13. Retrieval quality should be inspected before assuming the LLM is the source of a bad answer.**

---

# 📋 **One-Page Revision**

```text
                    INDEXING
                       │
PDF
 ↓
PyPDFLoader
 ↓
Documents
 ↓
RecursiveCharacterTextSplitter
 ↓
Chunks
 ↓
OpenAIEmbeddings
 ↓
Vectors
 ↓
Chroma
                       │
                       ▼
                  VECTOR DB
                       │
                       │
                    RUNTIME
                       │
                       ↓
                  User Query
                       ↓
               Query Embedding
                       ↓
                 Similarity Search
                       ↓
                Relevant Chunks
                       ↓
                    Context
                       ↓
              Query + Context
                       ↓
                    Prompt
                       ↓
                      LLM
                       ↓
                    Answer
```

---

# 🎯 **Final Mental Model**

> **RAG implementation is the architecture translated into code:**

```text
LOAD
 ↓
SPLIT
 ↓
EMBED
 ↓
STORE
 ↓
RETRIEVE
 ↓
AUGMENT
 ↓
GENERATE
```

**The full conceptual formula is:**

$$
\text{RAG}
=
\text{Indexing}
+
\text{Retrieval}
+
\text{Augmentation}
+
\text{Generation}
$$

**where:**

```text
Indexing
→ Load + Chunk + Embed + Store

Retrieval
→ Find relevant chunks

Augmentation
→ Query + Retrieved Context

Generation
→ LLM Answer
```

---

# 🔗 **Useful Resource**

> **LangChain APIs evolve over time. Check the current official documentation for recommended packages and APIs when implementing a new RAG application.**

```text
LangChain Documentation
https://docs.langchain.com
```

---

# 🏁 **End Note**

```text
Knowledge Source
      ↓
Index Once
      ↓
Vector Database
      ↓
User Query
      ↓
Retrieve Relevant Knowledge
      ↓
Add Knowledge to Query
      ↓
LLM
      ↓
Answer
```

> **The most important RAG implementation idea is: do the document preparation during indexing, then retrieve only the relevant knowledge at runtime before generating the answer.**
